# NYU Paris - Numerical Analysis

Before you turn in this notebook, make sure everything runs as expected. First, **restart the kernel** and then **run all cells**. 
Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE".

## Neural networks: an interpolation viewpoint

In this lab, we see neural networks as a natural extension of piecewise linear interpolation,
and we train them with the [Flux](https://fluxml.ai) package.

| Part | Topic | Suggested time |
|------|-------|----------------|
| 1 | Piecewise linear interpolation with ReLU functions | 25 min |
| 2 | Learning the breakpoints with Flux | 30 min |
| 3 | The learning rate, and the Runge function | 20 min |
| 4 | Overfitting | 20 min |
| 5 | Classifying points in the plane | 25 min |
| 6 | Handwritten digits | 25 min |

**On using AI assistants.**
You are welcome to use an AI assistant for the "plumbing" (plotting, Julia syntax).
However, you should write the core of each exercise yourself, and be able to explain every line of it.

If Flux or ChainPlots are not installed, run `import Pkg; Pkg.add(["Flux", "ChainPlots"])` once (this takes a few minutes).

In [ ]:
using ChainPlots
using Flux
using LaTeXStrings
using Plots
using Polynomials
using Random

### <font color='orange'>[Exercise 1]</font> Piecewise linear interpolation with ReLU functions

The *rectified linear unit* (ReLU) is the function $σ(x) = \max(0, x)$.
In this exercise, we show that piecewise linear functions are combinations of ReLU functions,
and we compute such combinations by least squares, as in the chapter on approximation.

1. Complete the function `hat` below so that it returns
   $$
   φ(x) = σ(x + 1) - 2σ(x) + σ(x - 1),
   $$
   and plot it on the interval $[-2, 2]$. What do you recognize?

In [ ]:
σ(x) = max(0, x)

function hat(x)
    # YOUR CODE HERE
end

# YOUR CODE HERE

In [ ]:
@assert hat(0) == 1
@assert hat(0.5) == 0.5
@assert hat(1.5) == 0

2. More generally, any continuous piecewise linear function with breakpoints $t_1 < \dots < t_m$
   can be written, for $x \geq t_1$, as
   $$
   \hat u(x) = c + \sum_{j=1}^{m} a_j σ(x - t_j).
   $$
   We approximate the Runge function $u(x) = 1/(1 + 25x^2)$ by such a function,
   with $m = 9$ breakpoints equally spaced in $[-1, 1)$.
   The coefficients $(c, a_1, \dots, a_m)$ are obtained by least squares from $n = 50$ data points $(x_i, u(x_i))$.

   Complete the function `relu_lsq(m, xs)`, which
   constructs the matrix $\mathsf A \in \mathbb R^{n \times (m+1)}$ such that $\mathsf A_{i1} = 1$ and $\mathsf A_{i,j+1} = σ(x_i - t_j)$,
   solves the least squares problem with `A \ b`, and returns the approximation $\hat u$ as a function.
   Then plot $\hat u$ together with $u$.

In [ ]:
u(x) = 1 / (1 + 25x^2)
xs = LinRange(-1, 1, 50)

function relu_lsq(m, xs)
    t = LinRange(-1, 1, m + 1)[1:m]
    # YOUR CODE HERE
    return x -> α[1] + sum(α[j+1] * σ(x - t[j]) for j in 1:m)
end

# YOUR CODE HERE

3. Repeat the previous item for $m \in \{4, 8, 16, 32, 64\}$, with $n = 4m$ data points, and plot the maximum error
   $\max_{x \in [-1, 1]} \lvert u(x) - \hat u(x) \rvert$ as a function of $m$ on a log-log scale.
   How does the error decrease? Compare with what you know about piecewise linear interpolation.

In [ ]:
xfine = LinRange(-1, 1, 1000)

# YOUR CODE HERE

### <font color='orange'>[Exercise 2]</font> Learning the breakpoints with Flux

In Exercise 1, the breakpoints $t_j$ were fixed, so that the problem was *linear* in the unknowns.
A *neural network* with one hidden layer of width $m$ also learns the breakpoints:
$$
N_θ(x) = c + \sum_{j=1}^{m} a_j σ(w_j x + b_j),
$$
where the parameters are $θ = (w_j, b_j, a_j, c)$.
The breakpoints are then at $x = -b_j / w_j$.

With Flux, this network is written `Chain(Dense(1 => m, σ), Dense(m => 1))`:

- `Dense(1 => m, σ)` is the function $x \mapsto σ.(\mathbf w x + \mathbf b)$, with $\mathbf w, \mathbf b \in \mathbb R^m$;
- `Dense(m => 1)` is the function $\mathbf y \mapsto \mathbf a^T \mathbf y + c$;
- `Chain` composes the two.

The package `ChainPlots` draws a network, with one disk per neuron and one line per weight.
Flux works with *batches* of data: the inputs are stored in a $1 \times n$ matrix, one column per data point.
The function `row` below converts a vector to this format.

In [ ]:
row(v) = reshape(Float32.(collect(v)), 1, :)

Random.seed!(0)
model = Chain(Dense(1 => 8, relu), Dense(8 => 1))
display(plotchain(model))
x = row(LinRange(-1, 1, 5))
@show model(x)           # A 1 × 5 matrix
@show model[1].weight    # The vector w (as an 8 × 1 matrix)
@show model[1].bias;     # The vector b

Training the network means minimizing the *mean squared error*
$$
J(θ) = \frac{1}{n} \sum_{i=1}^{n} \bigl(N_θ(x_i) - u_i\bigr)^2,
$$
which is no longer a quadratic function of $θ$.
We use an iterative method: *gradient descent* $θ_{k+1} = θ_k - λ \nabla J(θ_k)$, or an improved version of it.
Flux computes the gradient $\nabla J$ for us, by *backpropagation*.

1. Complete the function `train!` below, which performs `nsteps` iterations of an optimization method `opt`
   and returns the vector of the values $J(θ_k)$.
   <details>
       <summary>
           <em><font color='gray'>Hint (click to display)</font></em>
       </summary>

   - `loss, grads = Flux.withgradient(m -> Flux.mse(m(x), y), model)` computes $J$ and its gradient;
   - `Flux.update!(state, model, grads[1])` performs one step of the method.
   </details>

In [ ]:
function train!(model, x, y; opt=Adam(0.01), nsteps=1000)
    state = Flux.setup(opt, model)
    losses = Float64[]
    for k in 1:nsteps
        # YOUR CODE HERE
    end
    return losses
end

2. We approximate the function $u(x) = e^{-100x^2}$ with $m = 8$ neurons.
   Train the network below on the data `x`, `y` with `Adam(0.005)` and 5000 iterations,
   and plot the loss as a function of the iteration number, with a logarithmic scale for the y-axis.
   The function `init_breakpoints!` spreads the initial breakpoints uniformly in $[-1, 1]$.

In [ ]:
bump(x) = exp(-100x^2)
xtrain = LinRange(-1, 1, 200)
x, y = row(xtrain), row(bump.(xtrain))

function init_breakpoints!(model)
    m = length(model[1].bias)
    model[1].weight .= rand([-1f0, 1f0], m, 1)
    model[1].bias .= -model[1].weight[:, 1] .* Float32.(LinRange(-0.9, 0.9, m))
    return model
end

Random.seed!(1)
model = init_breakpoints!(Chain(Dense(1 => 8, relu), Dense(8 => 1)))

# YOUR CODE HERE

3. Plot the function $u$, the trained network, and the breakpoints $-b_j / w_j$ of the network.
   Compare with the piecewise linear interpolant with 8 pieces of equal length.
   Where are the breakpoints of the network?

In [ ]:
# YOUR CODE HERE

### <font color='orange'>[Exercise 3]</font> The learning rate, and the Runge function

We now use the activation function $\tanh$ instead of the ReLU, and we fit the Runge function from 15 equally spaced points.

1. Train the network `Chain(Dense(1 => 20, tanh), Dense(20 => 1))` with gradient descent, `opt = Descent(λ)`,
   for 2000 iterations and $λ \in \{0.05, 0.2, 0.5, 0.7\}$, always starting from the same initial parameters.
   Plot the four loss curves on the same graph. What do you observe?
   Relate your observations to the convergence of Richardson's iteration for linear systems.

In [ ]:
xs = LinRange(-1, 1, 15)
x, y = row(xs), row(u.(xs))

# YOUR CODE HERE

2. Train the same network with `Adam(0.01)` and 5000 iterations.
   Plot it together with the Runge function and the polynomial interpolating the data.
   Which approximation is better? Is the comparison fair?
   <details>
       <summary>
           <em><font color='gray'>Hint (click to display)</font></em>
       </summary>

   The interpolating polynomial can be computed with `fit(xs, u.(xs))`, from the `Polynomials` package.
   </details>

In [ ]:
# YOUR CODE HERE

### <font color='orange'>[Exercise 4]</font> Overfitting

In practice, data are noisy. Below, we sample $n = 20$ noisy values of $u(x) = \sin(\pi x)$,
and we keep 200 additional noiseless points to evaluate the networks: the *test set*.

In [ ]:
Random.seed!(3)
xn = sort(2rand(20) .- 1)
yn = sin.(π * xn) + 0.15randn(20)
xtest = LinRange(-1, 1, 200)
x, y = row(xn), row(yn)
x_test, y_test = row(xtest), row(sin.(π * xtest))
scatter(xn, yn, label="Noisy data")
plot!(x -> sin(π * x), -1, 1, label=L"\sin(\pi x)")

We compare a large and a small network:

In [ ]:
Random.seed!(4)
large = Chain(Dense(1 => 64, tanh), Dense(64 => 64, tanh), Dense(64 => 1))
small = Chain(Dense(1 => 4, tanh), Dense(4 => 1))
plot(plotchain(large, title="Large network"), plotchain(small, title="Small network"), size=(900, 350))

1. Train the large network for 20000 iterations and the small network for 3000 iterations, both with `Adam(0.01)`.
   For each network, compute the number of parameters (with `sum(length, Flux.trainables(model))`),
   the error on the training data, and the error on the test set (with `Flux.mse`).
   Plot both networks. Which one is better?

In [ ]:
# YOUR CODE HERE

2. Train a new large network for 20000 iterations, and record both the training error and the test error
   every 100 iterations. Plot both curves. When should the training have been stopped?

In [ ]:
# YOUR CODE HERE

### <font color='orange'>[Exercise 5]</font> Classifying points in the plane

Networks also work in several dimensions.
Here, each point of the plane belongs to one of two intertwined spirals, labelled $+1$ and $-1$.
We train a network $N_θ\colon \mathbb R^2 \to \mathbb R$ to fit the labels,
and we predict the label of a point by the sign of $N_θ$.

In [ ]:
function spirals(n; noise=0.1, rng=MersenneTwister(1))
    X, Y = zeros(Float32, 2, 2n), zeros(Float32, 1, 2n)
    for i in 1:n, (j, s) in enumerate((1, -1))
        t = sqrt(rand(rng)) * 3π
        X[:, 2(i-1)+j] = s * t * [cos(t), sin(t)] / 3π + noise / 3π * randn(rng, 2)
        Y[1, 2(i-1)+j] = s
    end
    return X, Y
end

X, Y = spirals(300)
scatter(X[1, :], X[2, :], zcolor=vec(Y), c=:coolwarm, aspect_ratio=:equal, label=nothing, colorbar=false)

1. Train a network with two hidden layers of 32 neurons and the activation function $\tanh$
   for 3000 iterations, with `Adam(0.01)`.
   Compute the fraction of correctly classified points.

In [ ]:
# YOUR CODE HERE

2. Plot the *decision boundary*, i.e. the curve where $N_θ = 0$, together with the data.
   <details>
       <summary>
           <em><font color='gray'>Hint (click to display)</font></em>
       </summary>

   Evaluate the network on a grid, e.g. `Z = [model([x, y])[1] for y in grid, x in grid]`,
   and use `contourf(grid, grid, Z)` and `contour!(grid, grid, Z, levels=[0])`.
   </details>

In [ ]:
# YOUR CODE HERE

3. Try a network with a single hidden layer of 32 neurons, and one with three hidden layers of 8 neurons.
   Draw the three networks with `plotchain`, and compare their numbers of parameters and their accuracies.

In [ ]:
# YOUR CODE HERE

### <font color='orange'>[Exercise 6]</font> Handwritten digits

The MNIST dataset contains 70 000 grayscale images of handwritten digits, of $28 \times 28$ pixels:
60 000 *training* images and 10 000 *test* images.
We see each image as a vector $x \in [0, 1]^{784}$, with one entry per pixel.
Recognizing digits then amounts to approximating a function $f\colon [0, 1]^{784} \to \{0, \dots, 9\}$
from its values at the 60 000 training images: this is an interpolation problem in dimension 784!

The data (about 10 MB) are downloaded by the `MLDatasets` package.
If it is not installed, run `import Pkg; Pkg.add("MLDatasets")` once.

In [ ]:
using MLDatasets
ENV["DATADEPS_ALWAYS_ACCEPT"] = true  # Download the data without asking for confirmation

train_data, test_data = MNIST(split=:train), MNIST(split=:test)
Xtrain = reshape(train_data.features, 784, :)  # A 784 × 60000 matrix, one column per image
Xtest = reshape(test_data.features, 784, :)
ytrain, ytest = train_data.targets, test_data.targets

show_digit(x; kwargs...) = heatmap(reshape(x, 28, 28)', yflip=true, c=:grays, aspect_ratio=:equal,
                                   ticks=false, axis=false, colorbar=false; kwargs...)
plot([show_digit(Xtrain[:, i], title=ytrain[i]) for i in 1:12]..., layout=(2, 6), size=(900, 350))

1. The simplest method is the *nearest neighbour* classifier:
   the predicted digit for an image $x$ is the label of the training image closest to $x$ in the Euclidean norm.
   In other words, $f$ is approximated by a *piecewise constant* interpolant,
   which is constant on the set of points closest to each training image (its *Voronoi cell*).

   Complete the function `nearest_neighbour(X, Xtrain, ytrain)`,
   which returns the vector of predicted labels for the images in the columns of `X`.
   Then compute the fraction of correctly classified images among the first 1000 test images.
   <details>
       <summary>
           <em><font color='gray'>Hint (click to display)</font></em>
       </summary>

   Since $\lVert x - z \rVert^2 = \lVert x \rVert^2 - 2 x^T z + \lVert z \rVert^2$,
   the matrix of the squared distances between the columns of `Xtrain` and those of `X` can be computed with a single matrix product,
   `Xtrain' * X`. Avoid computing the distances one by one: this would be very slow.
   </details>

In [ ]:
function nearest_neighbour(X, Xtrain, ytrain)
    # YOUR CODE HERE
end

In [ ]:
@assert nearest_neighbour([0 2.9; 0 0.1], [0 1 3; 0 0 0], [7, 8, 9]) == [7, 9]

In [ ]:
Xt, yt = Xtest[:, 1:1000], ytest[1:1000]

# YOUR CODE HERE

2. Compute the error rate (the fraction of misclassified images among the first 1000 test images)
   when only the first $n$ training images are used,
   for $n \in \{500, 1000, 2000, 5000, 10000, 20000, 60000\}$,
   and plot it as a function of $n$ on a log-log scale.
   Estimate the rate of convergence. Compare with Exercise 1: why is the convergence so slow?
   <details>
       <summary>
           <em><font color='gray'>Hint (click to display)</font></em>
       </summary>

   In dimension $d$, the typical distance between $n$ points spread uniformly in a cube is of order $n^{-1/d}$.
   </details>

In [ ]:
# YOUR CODE HERE

3. We now train the network `Chain(Dense(784 => 64, relu), Dense(64 => 10))`.
   It has 10 outputs, one score per digit, and the predicted digit is the one with the largest score,
   given by `Flux.onecold(model(X), 0:9)`.
   For training, each label is encoded as a vector of $\mathbb R^{10}$ with a single nonzero entry (`Flux.onehotbatch(y, 0:9)`),
   and the mean squared error is replaced by the *cross-entropy* `Flux.logitcrossentropy`, which is better suited to classification.

   With 60 000 images, computing the gradient of the loss over the whole training set at each iteration would be expensive.
   Instead, each iteration uses a *mini-batch* of 128 images, drawn at random by `Flux.DataLoader`:
   this is *stochastic gradient descent*. One pass over the training set is called an *epoch*.

   Complete the training loop below, and train the network for 5 epochs.
   How do the training and test accuracies evolve?

In [ ]:
digit_accuracy(model, X, y) = sum(Flux.onecold(model(X), 0:9) .== y) / length(y)

Random.seed!(1)
model = Chain(Dense(784 => 64, relu), Dense(64 => 10))
loader = Flux.DataLoader((Xtrain, Flux.onehotbatch(ytrain, 0:9)), batchsize=128, shuffle=true)
state = Flux.setup(Adam(0.001), model)
for epoch in 1:5
    for (x, y) in loader
        # YOUR CODE HERE
    end
    println("Epoch $epoch: training accuracy $(digit_accuracy(model, Xtrain, ytrain)), ",
            "test accuracy $(digit_accuracy(model, Xtest, ytest))")
end

4. Compare the network with the nearest neighbour classifier:
   accuracy, amount of data to store (the parameters of the network, or the training images),
   and cost of classifying one image.
   Then plot a few test images that the network misclassifies, with their true and predicted labels.
   Would you have recognized them?

In [ ]:
# YOUR CODE HERE

### Going further

If you finish early, here are a few ideas.

- **Depth.** The tent function $g(x) = 2σ(x) - 4σ(x - 1/2)$ uses two neurons.
  Plot $g \circ g \circ \dots \circ g$ ($k$ times) on $[0, 1]$, and count its linear pieces.
  How many neurons would a network with one hidden layer need to represent it?
- **Painting with a network.** An image is a function $(x, y) \mapsto$ gray level.
  Train a network $\mathbb R^2 \to \mathbb R$ on the pixels of a picture of your choice. Is the result sharp?
  Look up *Fourier features* to improve it.
- **Your own digits.** Draw a digit yourself, as a $28 \times 28$ image (white on black),
  and check whether the network of Exercise 6 recognizes it.
  The digits of MNIST are centred and scaled: does it matter?
- **Under the hood.** How does Flux compute gradients? Read about *reverse-mode automatic differentiation*,
  and compare with the dual numbers seen in class.